# Bronze Layer: Raw FHIR API Data Ingestion

# Overview
This notebook fetches raw FHIR resources (`Patient`, `Encounter`, `Observation`, `Condition`) from the HAPI FHIR REST API.

# Key Steps & Features
 **API Pagination**: Automatically handles pagination up to 3 pages per resource using the `next` link relation.
 **Audit Metadata**: Injects ingestion metadata including `extraction_timestamp` and `api_url_or_params`.
 **Sequential Ingestion**: Processes endpoints in the mandated order:
  1. `Patient`
  2. `Encounter`
  3. `Observation`
  4. `Condition`
 **Storage**: Saves raw JSON payloads directly into Bronze Delta Lake tables.

In [0]:
import requests
import json
from datetime import datetime, timezone
from pyspark.sql.functions import current_timestamp, lit

# 1. Clean up old conflicting tables if they exist
old_tables = ["bronze_patient", "bronze_encounter", "bronze_observation", "bronze_condition"]
for t in old_tables:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

BASE_URL = "https://hapi.fhir.org/baseR4"

def fetch_and_ingest_fhir(resource_type: str, max_pages: int = 3):
    """
    Fetches FHIR data with pagination, adds metadata, and saves to Bronze Delta Table.
    """
    print(f"--- Processing {resource_type} ---")
    today_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    api_url = f"{BASE_URL}/{resource_type}?_count=50&_sort=-_lastUpdated"
    page_num = 1
    
    while api_url and page_num <= max_pages:
        try:
            response = requests.get(api_url, timeout=30)
            if response.status_code != 200:
                print(f"Failed to fetch {resource_type} page {page_num}: Status {response.status_code}")
                break
                
            data = response.json()
            entries = data.get("entry", [])
            
            if not entries:
                print(f"No more entries found for {resource_type}.")
                break

            # Convert JSON payload into PySpark DataFrame
            json_str = json.dumps(data)
            df_raw = spark.createDataFrame([(json_str,)], ["raw_payload"])
            
            # Add assignment required Metadata
            df_bronze = df_raw.withColumn("extraction_timestamp", current_timestamp()) \
                              .withColumn("api_url_or_params", lit(api_url))
            
            # Save into Bronze Delta Table (overwrite mode for page 1, append for page 2+)
            table_name = f"bronze_{resource_type.lower()}"
            write_mode = "overwrite" if page_num == 1 else "append"
            
            df_bronze.write \
                .format("delta") \
                .mode(write_mode) \
                .option("overwriteSchema", "true") \
                .saveAsTable(table_name)
            
            print(f"Page {page_num} ingested successfully into {table_name}.")

            # Extract next pagination link
            next_link = next((link["url"] for link in data.get("link", []) if link.get("relation") == "next"), None)
            api_url = next_link
            page_num += 1

        except Exception as e:
            print(f"Error fetching {resource_type}: {e}")
            break

# Ingest in mandated order: Patient -> Encounter -> Observation -> Condition
fhir_resources = ["Patient", "Encounter", "Observation", "Condition"]
for resource in fhir_resources:
    fetch_and_ingest_fhir(resource)

--- Processing Patient ---
Page 1 ingested successfully into bronze_patient.
Page 2 ingested successfully into bronze_patient.
Page 3 ingested successfully into bronze_patient.
--- Processing Encounter ---
Page 1 ingested successfully into bronze_encounter.
Page 2 ingested successfully into bronze_encounter.
Page 3 ingested successfully into bronze_encounter.
--- Processing Observation ---
Page 1 ingested successfully into bronze_observation.
Page 2 ingested successfully into bronze_observation.
Page 3 ingested successfully into bronze_observation.
--- Processing Condition ---
Page 1 ingested successfully into bronze_condition.
Page 2 ingested successfully into bronze_condition.
Page 3 ingested successfully into bronze_condition.
